# Lab 1 · Exercise C — signal + background, extended fit
Data: invariant mass of candidates in a search for a rare $B^0$ decay (`examples/lhcb-b0-ppbar/data/rarest_b0_decay.dat`, 166 events, MeV).
Inspired by LHCb, *First observation of the rare purely baryonic decay $B^0 \to p\bar p$*.

In [ ]:
import ROOT
%jsroot on   # interactive drawing in the notebook (needed on Binder)
import numpy as np

MATRICOLA = 0   # TODO: write your student ID number here (digits only)
assert MATRICOLA != 0, "Set MATRICOLA first: it seeds YOUR random toy data"

ROOT.RooRandom.randomGenerator().SetSeed(MATRICOLA)
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.WARNING)

def check(name, ok):
    """Minimal self-check: prints PASS/FAIL and stops the notebook on FAIL."""
    print(("PASS  " if ok else "FAIL  ") + name)
    assert ok, name

In [ ]:
vals = np.loadtxt("../../examples/lhcb-b0-ppbar/data/rarest_b0_decay.dat")
print(len(vals), "events, range", vals.min(), vals.max())

m = ROOT.RooRealVar("m", "m [MeV]", 5080, 5540)
data = ROOT.RooDataSet("data", "data", ROOT.RooArgSet(m))
for v in vals:
    m.setVal(v)
    data.add(ROOT.RooArgSet(m))
check("dataset filled", data.numEntries() == len(vals))

## C1 · Extended ML fit
Signal: Gaussian (mean near 5280 MeV). Background: exponential. Yields `nsig`, `nbkg` are fit parameters.
Report N_s and N_b with errors; plot data, total, and the background component; add the pulls.

In [ ]:
mean  = ROOT.RooRealVar("mean", "mean", 5280, 5200, 5350)
sigma = ROOT.RooRealVar("sigma", "sigma", 15, 5, 50)
slope = ROOT.RooRealVar("slope", "slope", -0.002, -0.02, 0)
sig   = None   # TODO: RooGaussian(m, mean, sigma)
bkg   = None   # TODO: RooExponential(m, slope)

nsig  = ROOT.RooRealVar("nsig", "signal yield", 30, 0, 166)
nbkg  = ROOT.RooRealVar("nbkg", "background yield", 130, 0, 500)
model = None   # TODO: RooAddPdf of sig and bkg with yields nsig, nbkg (RooArgList for both)
res   = None   # TODO: extended ML fit (Extended=True, Save=True, PrintLevel=-1)

frame = m.frame(Title="Signal + background", Bins=23)
# TODO: plot data, model, and the background component:  Components="bkg", LineStyle=ROOT.kDashed
c = ROOT.TCanvas("c", "c", 800, 600)
frame.Draw()
c.Draw()
nsig.Print(); nbkg.Print()

In [ ]:
# Pulls: (data - model) / error, per bin
pull_frame = m.frame(Title="Pulls", Bins=23)
# TODO: get the pull histogram from `frame` (frame.pullHist()) and add it to pull_frame (addPlotable(h, "P"))
cp = ROOT.TCanvas("cp", "cp", 800, 300)
pull_frame.Draw()
cp.Draw()

In [ ]:
check("model built", model is not None and res is not None)
check("fit converged", res.status() == 0)
check("nsig positive and with a finite error", nsig.getVal() > 0 and nsig.getError() > 0)
check("nsig + nbkg close to the number of events", abs(nsig.getVal() + nbkg.getVal() - len(vals)) < 5)
check("signal mass near the B0 mass (5279 MeV)", abs(mean.getVal() - 5279) < 40)

## C2 · Same exercise, shape-only (fractions)
Replace the yields with one fraction `fsig`. Compute N_s = fsig × 166 and its error, and compare with C1.
**What happened to the information on the total yield?**

In [ ]:
mean.setVal(5280); sigma.setVal(15); slope.setVal(-0.002)
fsig    = ROOT.RooRealVar("fsig", "signal fraction", 0.2, 0, 1)
model_f = None   # TODO: RooAddPdf of sig and bkg with ONE coefficient (fsig)
res_f   = None   # TODO: ML fit (no Extended), Save=True, PrintLevel=-1
# TODO: print N_s = fsig * len(vals) with its error, and the N_s from C1

## C4 · (optional, 10 min) RDataFrame with a C++ function, compiled just in time
Count the events in the mass window 5200–5360 MeV with an `RDataFrame` and a **C++ function** declared from Python.
Compare with numpy: the two numbers must agree.

In [ ]:
ROOT.gInterpreter.Declare("""
bool in_window(double m, double lo, double hi) { return m > lo && m < hi; }
""")
rdf = ROOT.RDF.FromNumpy({"m": vals})
n_win = None   # TODO: rdf.Filter("in_window(m, 5200., 5360.)").Count().GetValue()
print(n_win, "events in the window (RDataFrame);", int(np.sum((vals > 5200) & (vals < 5360))), "with numpy")

In [ ]:
check("RDataFrame (C++ JIT) agrees with numpy", n_win == int(np.sum((vals > 5200) & (vals < 5360))))

**Questions C.** (a) Which error is larger, and why? (b) In which analysis would you *need* the rate?
(c) Is the signal significant? Do not compute a p-value yet: just compare N_s with its error. We will do it properly in Lab 3.